# token-class

Assign a label to every token. This folder is a drop-in trainer for any token-classification dataset: swap the example in `preprocess.py` and `config.json`, then run.

## Example: emotion spans on GoEmotions

The bundled example finds emotion *spans* in Reddit comments from [GoEmotions](https://huggingface.co/datasets/google-research-datasets/go_emotions), using [sdeakin/GoEmotions-Projected-BIO-Emotions](https://huggingface.co/datasets/sdeakin/GoEmotions-Projected-BIO-Emotions). Each row is a comment plus character/token spans labeled with an emotion (`joy`, `sadness`, `anger`, …).

`preprocess.py` turns those spans into per-token BIO tags:

| token | tag |
| --- | --- |
| I | `O` |
| am | `O` |
| so | `B-Joy` |
| happy | `I-Joy` |
| today | `O` |

- `B-{emotion}` starts a span, `I-{emotion}` continues it, `O` is outside any span.
- Emotion names are taken from each span's `subtype` (or `type`) and normalized to `Joy`, `Sadness`, etc. Spans missing an emotion label or token indices are skipped.
- Empty trailing tokens are stripped after BIO tags are assigned (span indices refer to the original token list); comments with no tokens are dropped.
- Labels are collected from the data (`O` first, then every `B-*` / `I-*` tag that appears).
- The split is 90/10 train/val (`test_size` and `seed` in `config.json`).

Training fine-tunes `distilbert-base-uncased` as a token classifier. The base checkpoint has no token-classification head, so Hugging Face initializes `classifier.*` randomly (the MLM head weights are unused). Wordpiece tokens inherit the word's label; subword continuations and special tokens are ignored (`-100`) so they do not affect loss or metrics.

Warmup is `warmup_ratio` of the Trainer's optimizer-update count, converted to `warmup_steps` after the dataloader is built (so GPU count and `gradient_accumulation_steps` are included). The first `span_f1` is reported at the end of epoch 1 (`eval_strategy` is `epoch`). Training stops early if `span_f1` does not improve for `early_stopping_patience` epochs; the best checkpoint is saved. Saved weights include both `LayerNorm.weight`/`bias` and `gamma`/`beta` names so DistilBERT reloads cleanly.

`evaluate.py` reports:

- **token accuracy** — share of labeled word tokens predicted correctly
- **span precision / recall / F1** — exact match on `(start, end, emotion)` spans, which is the metric used to pick the best checkpoint (`span_f1`)

`infer.py` prints a few validation comments that contain at least one gold span, with gold vs predicted spans and a token-level diff (`!` on mismatches).

## Layout

```
.
├── config.json    # model, data URL, split, and training settings
├── preprocess.py  # load GoEmotions spans and build BIO labels
├── train.py       # train and save the best checkpoint
├── evaluate.py    # score the saved model (token accuracy, span P/R/F1)
├── infer.py       # run the saved model on validation examples
├── README.md
```

Edit `config.json`, then from this folder:

```
python train.py
python evaluate.py
python infer.py
```


## Setup


In [ ]:
!pip install uv


In [ ]:
!uv pip install --system accelerate datasets numpy torch transformers


In [ ]:
from IPython.display import HTML, display
display(HTML("<style>pre,.output_text{white-space:pre-wrap!important;word-break:break-word!important}</style>"))


## Config


In [ ]:
%%writefile config.json
{
  "model": "distilbert-base-uncased",
  "max_length": 128,
  "seed": 42,
  "data_url": "https://huggingface.co/datasets/sdeakin/GoEmotions-Projected-BIO-Emotions/resolve/main/GoEmotions-Projected-BIO-Emotions.jsonl",
  "test_size": 0.1,
  "output_dir": "output",
  "num_train_epochs": 10,
  "per_device_train_batch_size": 16,
  "per_device_eval_batch_size": 32,
  "learning_rate": 5e-5,
  "weight_decay": 0.01,
  "warmup_ratio": 0.06,
  "eval_strategy": "epoch",
  "save_strategy": "epoch",
  "save_total_limit": 2,
  "logging_steps": 50,
  "metric_for_best_model": "span_f1",
  "early_stopping_patience": 2,
  "show_examples": 6
}


## Logs


In [ ]:
%%writefile progress.py
"""Keep tqdm/transformers logs readable when the output pane is narrow or resized.

Shared by every task. Import this before `datasets` or `transformers` so
progress bars are configured first.

Kaggle's log viewer (and most notebook captures) do not treat ``\\r`` as
"overwrite this line", and they do not expose the pane width as ``COLUMNS``.
A full-width tqdm bar then wraps mid-update and stacks into a staircase.
This module disables those bars in captured/Kaggle output and uses a compact,
``dynamic_ncols`` bar in a real terminal so a resize still fits. It also drops
a few known-harmless messages (DataParallel scalar-gather, and in captured
environments the DistilBERT load-mismatch table and unauthenticated Hub
notice) without changing framework log levels. Trainer log lines pad numeric
fields with zeros so loss, grad norm, learning rate, and epoch share a stable
width.
"""

from __future__ import annotations

import logging
import numbers
import os
import sys
import warnings
from pathlib import Path

_EXPECTED_LOG_NEEDLES = (
    "unauthenticated requests to the HF Hub",
    "LOAD REPORT",
)

_GATHER_WARNING = "Was asked to gather along dimension 0"
_log_filter: _ExpectedLogFilter | None = None
_gather_showwarning_wrapped = False


class _ExpectedLogFilter(logging.Filter):
    def filter(self, record: logging.LogRecord) -> bool:
        try:
            msg = record.getMessage()
        except Exception:
            return True
        return not any(needle in msg for needle in _EXPECTED_LOG_NEEDLES)


def captured_display() -> bool:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return True
    if Path("/kaggle/input").exists() or Path("/kaggle/working").exists():
        return True
    if os.environ.get("COLAB_RELEASE_TAG"):
        return True
    try:
        return not sys.stdout.isatty()
    except Exception:
        return True


def disable_tqdm() -> bool:
    return captured_display()


def _quiet_gather_warning() -> None:
    """Drop DataParallel's scalar-gather UserWarning even if filters are reset."""
    global _gather_showwarning_wrapped
    warnings.filterwarnings("ignore", message=r".*gather along dimension 0.*", category=UserWarning)
    if _gather_showwarning_wrapped:
        return
    original = warnings.showwarning

    def showwarning(message, category, filename, lineno, file=None, line=None):
        if _GATHER_WARNING in str(message) and issubclass(category, UserWarning):
            return
        return original(message, category, filename, lineno, file=file, line=line)

    warnings.showwarning = showwarning
    _gather_showwarning_wrapped = True


def configure() -> None:
    _quiet_gather_warning()
    if disable_tqdm():
        os.environ["TQDM_DISABLE"] = "1"
        os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
        os.environ.setdefault("PYDEVD_DISABLE_FILE_VALIDATION", "1")
        _install_expected_log_filter()
        try:
            from datasets.utils.logging import disable_progress_bar as disable_datasets_bar

            disable_datasets_bar()
        except Exception:
            pass
        try:
            from transformers.utils.logging import disable_progress_bar as disable_hf_bar

            disable_hf_bar()
        except Exception:
            pass
        return

    os.environ.setdefault("TQDM_DYNAMIC_NCOLS", "True")
    _patch_tqdm_for_resize()


def _in_filter_namespace(name: str) -> bool:
    return (
        name == "transformers"
        or name.startswith("transformers.")
        or name == "huggingface_hub"
        or name.startswith("huggingface_hub.")
    )


def _attach_expected_log_filter(logger: object) -> None:
    if _log_filter is None or not isinstance(logger, logging.Logger):
        return
    if not _in_filter_namespace(logger.name):
        return
    if _log_filter not in logger.filters:
        logger.addFilter(_log_filter)


def _install_expected_log_filter() -> None:
    global _log_filter
    if _log_filter is None:
        _log_filter = _ExpectedLogFilter()
    manager = logging.Logger.manager
    for logger in list(manager.loggerDict.values()):
        _attach_expected_log_filter(logger)
    for name in ("transformers", "huggingface_hub"):
        _attach_expected_log_filter(logging.getLogger(name))
    if not getattr(manager, "_easytrainer_filtered_getLogger", False):
        original = manager.getLogger

        def getLogger(name):
            logger = original(name)
            _attach_expected_log_filter(logger)
            return logger

        manager.getLogger = getLogger
        manager._easytrainer_filtered_getLogger = True


def _patch_tqdm_for_resize() -> None:
    try:
        import tqdm.std as std
    except Exception:
        return

    original_init = std.tqdm.__init__

    def __init__(self, *args, **kwargs):
        kwargs.setdefault("dynamic_ncols", True)
        kwargs.setdefault("mininterval", 1.0)
        kwargs.setdefault(
            "bar_format",
            "{desc}: {percentage:3.0f}% {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
        )
        original_init(self, *args, **kwargs)

    std.tqdm.__init__ = __init__


def format_log_number(key: str, value: object) -> object:
    """Pad numeric Trainer fields with leading/trailing zeros so columns line up."""
    if isinstance(value, bool) or not isinstance(value, numbers.Real):
        return value
    name = str(key).lower()
    if name == "epoch" or name.endswith("_epoch"):
        return f"{float(value):010.7f}"
    if "learning_rate" in name or name == "lr":
        return f"{float(value):0.4e}"
    if name in {"step", "global_step"} or name.endswith("_step") or name.endswith("_steps"):
        return f"{int(value):06d}"
    return f"{float(value):07.4f}"


def format_trainer_logs(logs: dict) -> dict:
    return {key: format_log_number(key, value) for key, value in logs.items() if key != "total_flos"}


def _pop_callback(trainer, callback_cls) -> None:
    try:
        trainer.pop_callback(callback_cls)
    except Exception:
        pass


def attach_aligned_logging(trainer) -> None:
    """Use zero-padded loss lines instead of PrinterCallback when tqdm is off.

    On a TTY, Trainer already logs through ProgressCallback; adding another
    printer would duplicate every ``on_log`` record.
    """
    from transformers.trainer_callback import PrinterCallback, TrainerCallback

    class AlignedLogCallback(TrainerCallback):
        def on_log(self, args, state, control, logs=None, **kwargs):
            if not logs or not getattr(state, "is_local_process_zero", True):
                return
            print(format_trainer_logs(logs), flush=True)

    _pop_callback(trainer, PrinterCallback)
    if disable_tqdm():
        trainer.add_callback(AlignedLogCallback())


def print_eval_metrics(trainer) -> dict:
    """Evaluate and print formatted metrics even if ``evaluate()`` never logs.

    Standalone ``Trainer.evaluate()`` only started calling ``self.log`` in
    transformers ~4.44. Always print the returned dict so older installs still
    show scores, and drop default printers so newer installs do not print twice.
    """
    from transformers.trainer_callback import PrinterCallback, ProgressCallback

    _pop_callback(trainer, PrinterCallback)
    _pop_callback(trainer, ProgressCallback)
    metrics = trainer.evaluate()
    if metrics:
        print(format_trainer_logs(metrics), flush=True)
    return metrics


configure()


## Preprocess


In [ ]:
%%writefile preprocess.py
"""
THIS FILE IS SPECIFICALLY FOR THE GOEMOTIONS BIO DATASET.

It loads sdeakin/GoEmotions-Projected-BIO-Emotions, converts emotion
spans into per-token BIO tags (B-Joy, I-Sadness, O, ...), and splits
train/val. Swap this file (and config.json) for any other
token-classification dataset.
"""

import json
import sys
from pathlib import Path

sys.path.append(str(Path(__file__).resolve().parents[1]))
import progress  # noqa: F401  # configure tqdm before datasets starts bars
from datasets import load_dataset

cfg = json.loads((Path(__file__).parent / "config.json").read_text())


def spans_to_bio(tokens, spans):
    tags = ["O"] * len(tokens)
    n = len(tokens)
    for span in spans:
        emotion = span.get("subtype") or span.get("type")
        start, end = span.get("start"), span.get("end")
        if not emotion or start is None or end is None or start < 0 or start >= n:
            continue
        emotion = emotion.replace(" ", "_").replace("-", "_")
        tags[start] = f"B-{emotion}"
        for i in range(start + 1, min(end, n - 1) + 1):
            tags[i] = f"I-{emotion}"
    return tags


def tag_spans(tags):
    spans, i = [], 0
    while i < len(tags):
        if tags[i].startswith("B-"):
            emotion, j = tags[i][2:], i + 1
            while j < len(tags) and tags[j] == f"I-{emotion}":
                j += 1
            spans.append((i, j, emotion))
            i = j
        else:
            i += 1
    return spans


def bio_to_spans(tokens, tags):
    return [(emo, " ".join(tokens[s:e])) for s, e, emo in tag_spans(tags)]


def to_example(row):
    tokens = list(row["data"]["tokens"])
    tags = spans_to_bio(tokens, row["data"].get("spans") or [])
    while tokens and tokens[-1] == "":
        tokens.pop()
        tags.pop()
    return {
        "text": row["text"],
        "tokens": tokens,
        "bio_tags": tags,
    }


raw = load_dataset("json", data_files=cfg["data_url"], split="train")
ds = raw.map(to_example, remove_columns=raw.column_names)
ds = ds.filter(lambda row: len(row["tokens"]))
split = ds.train_test_split(test_size=cfg["test_size"], seed=cfg["seed"])
train_ds, eval_ds = split["train"], split["test"]

labels = sorted({tag for tags in ds["bio_tags"] for tag in tags})
labels.remove("O")
labels = ["O"] + labels
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for name, i in label2id.items()}


## Evaluate


In [ ]:
%%writefile evaluate.py
"""Score a trained token classifier. Example: emotion-span BIO tags."""

import json
import sys
from pathlib import Path

sys.path.append(str(Path(__file__).resolve().parents[1]))
import progress
import numpy as np
import torch
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

from preprocess import eval_ds, id2label, label2id, tag_spans

root = Path(__file__).parent
cfg = json.loads((root / "config.json").read_text())


def make_tokenize(tokenizer):
    def tokenize(batch):
        encoded = tokenizer(
            batch["tokens"], is_split_into_words=True, truncation=True, max_length=cfg["max_length"]
        )
        aligned = []
        for i, tags in enumerate(batch["bio_tags"]):
            word_ids = encoded.word_ids(batch_index=i)
            ids, prev = [], None
            for word_id in word_ids:
                if word_id is None:
                    ids.append(-100)
                elif word_id != prev:
                    ids.append(label2id[tags[word_id]])
                else:
                    ids.append(-100)
                prev = word_id
            aligned.append(ids)
        encoded["labels"] = aligned
        return encoded

    return tokenize


def compute_metrics(eval_pred):
    logits, label_ids = eval_pred
    pred_ids = np.argmax(logits, axis=-1)
    gold_spans, pred_spans = [], []
    n_ok = n = 0
    offset = 0
    for pred_row, gold_row in zip(pred_ids, label_ids):
        gold, pred = [], []
        for p, g in zip(pred_row, gold_row):
            if g == -100:
                continue
            gold.append(id2label[int(g)])
            pred.append(id2label[int(p)])
            n_ok += int(p == g)
            n += 1
        gold_spans += [(s + offset, e + offset, emo) for s, e, emo in tag_spans(gold)]
        pred_spans += [(s + offset, e + offset, emo) for s, e, emo in tag_spans(pred)]
        offset += len(gold) + 1
    tp = len(set(gold_spans) & set(pred_spans))
    precision = tp / len(pred_spans) if pred_spans else 0.0
    recall = tp / len(gold_spans) if gold_spans else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "token_accuracy": n_ok / n,
        "span_precision": precision,
        "span_recall": recall,
        "span_f1": f1,
    }


if __name__ == "__main__":
    model_dir = str(root / cfg["output_dir"] / "best_model")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForTokenClassification.from_pretrained(model_dir)
    tokenized_eval = eval_ds.map(
        make_tokenize(tokenizer), batched=True, remove_columns=eval_ds.column_names
    )
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=str(root / cfg["output_dir"]),
            per_device_eval_batch_size=cfg["per_device_eval_batch_size"],
            fp16=torch.cuda.is_available(),
            report_to="none",
            disable_tqdm=progress.disable_tqdm(),
        ),
        eval_dataset=tokenized_eval,
        processing_class=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=compute_metrics,
    )
    progress.print_eval_metrics(trainer)


## Train


In [ ]:
%%writefile train.py
"""Train a token classifier. Example labels: emotion-span BIO tags."""

import json
import math
import os
import sys
from pathlib import Path

sys.path.append(str(Path(__file__).resolve().parents[1]))
import progress
import torch
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed,
)

from evaluate import compute_metrics, make_tokenize
from preprocess import eval_ds, id2label, label2id, labels, train_ds

root = Path(__file__).parent
cfg = json.loads((root / "config.json").read_text())
output_dir = str(root / cfg["output_dir"])

os.environ["WANDB_DISABLED"] = "true"
set_seed(cfg["seed"])

print(len(train_ds), "train,", len(eval_ds), "val,", len(labels), "labels")


def add_layernorm_aliases(state_dict):
    """DistilBERT checkpoints mix LayerNorm.weight/bias with gamma/beta names."""
    extra = {}
    pairs = (
        ("LayerNorm.gamma", "LayerNorm.weight"),
        ("LayerNorm.beta", "LayerNorm.bias"),
        ("_layer_norm.gamma", "_layer_norm.weight"),
        ("_layer_norm.beta", "_layer_norm.bias"),
    )
    for key, value in state_dict.items():
        for src, dst in pairs:
            if key.endswith(src):
                extra[f"{key[: -len(src)]}{dst}"] = value.detach().clone() if hasattr(value, "detach") else value
            elif key.endswith(dst):
                extra[f"{key[: -len(dst)]}{src}"] = value.detach().clone() if hasattr(value, "detach") else value
    state_dict.update(extra)
    return state_dict


def alias_saved_model(model_dir):
    model_dir = Path(model_dir)
    safetensors_path = model_dir / "model.safetensors"
    bin_path = model_dir / "pytorch_model.bin"
    if safetensors_path.exists():
        from safetensors.torch import load_file, save_file

        save_file(add_layernorm_aliases(dict(load_file(safetensors_path))), str(safetensors_path))
    elif bin_path.exists():
        torch.save(
            add_layernorm_aliases(torch.load(bin_path, map_location="cpu", weights_only=True)),
            bin_path,
        )


class LayerNormAliasCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        alias_saved_model(Path(args.output_dir) / f"checkpoint-{state.global_step}")


tokenizer = AutoTokenizer.from_pretrained(cfg["model"])
model = AutoModelForTokenClassification.from_pretrained(
    cfg["model"],
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)


tokenize = make_tokenize(tokenizer)
tokenized_train = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
tokenized_eval = eval_ds.map(tokenize, batched=True, remove_columns=eval_ds.column_names)

callbacks = [LayerNormAliasCallback()]
patience = cfg.get("early_stopping_patience")
if patience:
    callbacks.append(EarlyStoppingCallback(early_stopping_patience=patience))

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=cfg["num_train_epochs"],
        per_device_train_batch_size=cfg["per_device_train_batch_size"],
        per_device_eval_batch_size=cfg["per_device_eval_batch_size"],
        gradient_accumulation_steps=cfg.get("gradient_accumulation_steps", 1),
        learning_rate=cfg["learning_rate"],
        weight_decay=cfg["weight_decay"],
        eval_strategy=cfg["eval_strategy"],
        save_strategy=cfg["save_strategy"],
        save_total_limit=cfg.get("save_total_limit", 2),
        load_best_model_at_end=True,
        metric_for_best_model=cfg["metric_for_best_model"],
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=cfg["seed"],
        disable_tqdm=progress.disable_tqdm(),
        logging_steps=cfg.get("logging_steps", 50),
        logging_first_step=True,
    ),
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=callbacks,
)
progress.attach_aligned_logging(trainer)
train_loader = trainer.get_train_dataloader()
grad_accum = max(trainer.args.gradient_accumulation_steps, 1)
updates_per_epoch = max(1, math.ceil(len(train_loader) / grad_accum))
total_updates = math.ceil(trainer.args.num_train_epochs * updates_per_epoch)
# train() builds the scheduler from this; the dataloader above is only for step counts
trainer.args.warmup_steps = math.ceil(total_updates * cfg["warmup_ratio"])
trainer.train()
best_dir = Path(output_dir) / "best_model"
trainer.save_model(str(best_dir))
alias_saved_model(best_dir)


In [ ]:
!PYDEVD_DISABLE_FILE_VALIDATION=1 python -Xfrozen_modules=off train.py


In [ ]:
!PYDEVD_DISABLE_FILE_VALIDATION=1 python -Xfrozen_modules=off evaluate.py
